# Emerging Technologies
### Nathan Carr - G00410214

## Problem 1: Generating Random Boolean Functions

### Background

The Deutsch–Jozsa algorithm works with Boolean functions that take a fixed number of inputs and return a single Boolean output.

For this problem, the function takes **four Boolean inputs**, meaning there are:

\[
2^4 = 16
\]

possible input combinations.

The function is guaranteed to be either:

- **constant**: returns the same value (always `True` or always `False`)
- **balanced**: returns `True` for exactly half of the inputs (8 out of 16) and `False` for the rest

The goal is to generate a random function that satisfies this promise.

### Implementation

To solve this, I first generate all 16 possible input combinations using `itertools.product`.

Then:

- If the function is **constant**, I assign the same value (`True` or `False`) to all inputs.
- If the function is **balanced**, I randomly select 8 input combinations and assign them `True`, with the remaining inputs assigned `False`.

The function is stored as a lookup table (dictionary), and a Python function is returned that retrieves the output for a given input.

In [3]:
import random
from itertools import product

def random_constant_balanced(seed: int | None = None):
    """
    Return a random 4-input Boolean function that is either constant or balanced.

    Constant: always False or always True.
    Balanced: True for exactly 8 of the 16 possible inputs.
    """
    rng = random.Random(seed)
    inputs = list(product([False, True], repeat=4))  # 16 input tuples

    if rng.choice([True, False]):  # balanced
        true_inputs = set(rng.sample(inputs, k=8))   # choose exactly half
        table = {x: (x in true_inputs) for x in inputs}
    else:  # constant
        const_value = rng.choice([False, True])
        table = {x: const_value for x in inputs}

    def f(a: bool, b: bool, c: bool, d: bool) -> bool:
        return table[(a, b, c, d)]

    # Helpful for demonstration/testing in your notebook
    f.truth_table = table  # type: ignore[attr-defined]

    return f


### Demonstration

To verify that the function satisfies the required conditions, I count how many times it returns `True`.

For a valid function, the number of `True` outputs should always be:

- 0 (constant False)
- 16 (constant True)
- 8 (balanced)

In [4]:
# Generate a few functions and verify they match the promise
for s in range(5):
    f = random_constant_balanced(seed=s)
    num_true = sum(f.truth_table.values()) # type: ignore
    print(s, num_true)  # must be 0, 8, or 16


0 16
1 8
2 8
3 8
4 8


### Discussion

The results confirm that the generated function always satisfies the Deutsch–Jozsa promise.

This approach ensures correctness because:

- All possible inputs are explicitly considered
- The number of `True` outputs is controlled directly

This function will be used in later problems to compare classical and quantum approaches to identifying whether a function is constant or balanced.

### References

- IBM Quantum, *Deutsch–Jozsa Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa  
  This was used to understand the definition of constant and balanced Boolean functions.

- Python Documentation, *itertools.product*:  
  https://docs.python.org/3/library/itertools.html#itertools.product  
  This was used to generate all possible input combinations for a 4-bit Boolean function.

- Real Python, *Python Booleans*:  
  https://realpython.com/python-boolean/  
  This helped reinforce how Boolean values work in Python functions.

## Problem 2: Classical Testing for Function Type

### Background

In this problem, the goal is to determine whether a given Boolean function is **constant** or **balanced**.

The function is guaranteed to follow the same rules as in Problem 1:

- constant → always returns the same value  
- balanced → returns `True` for exactly half of the inputs  

Classically, we do this by calling the function with different inputs and observing the outputs.

### Implementation

To solve this, I evaluate the function on different input combinations.

- I store the result of the first function call.
- Then I keep checking new inputs:
  - If I ever see a different output, the function must be **balanced**.
  - If I continue seeing the same output, I keep checking until I can be certain it is **constant**.

Since there are 16 possible inputs, I loop through them systematically using `itertools.product`.

In [5]:
from itertools import product

def determine_constant_balanced(f) -> str:
    """
    Determine whether a promised constant/balanced 4-input Boolean function is
    "constant" or "balanced".

    Worst-case calls to f: 9 (guarantees 100% certainty under the promise).
    """
    first = None

    for i, x in enumerate(product([False, True], repeat=4), start=1):
        y = f(*x)

        if first is None:
            first = y
        elif y != first:
            return "balanced"

        # After 9 identical outputs, the function cannot be balanced (only 8 of each)
        if i == 9:
            return "constant"

    # With the promise, execution should always return before this.
    raise RuntimeError("Promise violated: function is neither constant nor balanced.")


### Efficiency

To be **100% certain**, a classical algorithm may need to evaluate the function multiple times.

In the worst case:

- A balanced function has 8 `True` and 8 `False` outputs.
- It is possible to observe the same output for the first 8 inputs and still not know if the function is constant or balanced.

Because of this, we must check **one more input** to be certain.

So the maximum number of function calls required is:

\[
2^{n-1} + 1
\]

For \(n = 4\):

\[
2^3 + 1 = 9
\]

This means a classical solution may require up to **9 evaluations** to guarantee the correct answer.

### Demonstration

To test the solution, I generate functions using the method from Problem 1 and apply the classification function.

The output shows:

- the number of `True` values (0, 8, or 16)
- the predicted result (`constant` or `balanced`)

This confirms that the function correctly identifies the type in all cases.

In [6]:
# Quick demo using Problem 1 generator
for s in range(6):
    f = random_constant_balanced(seed=s)
    print(f"seed={s:2d}  trues={sum(f.truth_table.values()):2d}  classified={determine_constant_balanced(f)}") # type: ignore

seed= 0  trues=16  classified=constant
seed= 1  trues= 8  classified=balanced
seed= 2  trues= 8  classified=balanced
seed= 3  trues= 8  classified=balanced
seed= 4  trues= 8  classified=balanced
seed= 5  trues=16  classified=constant


### Discussion

This approach works well because it stops early if a different output is found.

However, in the worst case, it still needs multiple evaluations to be certain.

This highlights an important limitation of classical computation:
we may need to check several inputs before knowing the answer.

In the next problems, this will be compared with quantum algorithms,
which can determine the result using fewer evaluations.

### References

- IBM Quantum, *Deutsch's Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm  
  This helped explain the classical problem that the quantum algorithm is designed to improve.

- IBM Quantum, *Deutsch–Jozsa Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa  
  This provided context on the difference between classical and quantum approaches.

- Python Documentation, *itertools.product*:  
  https://docs.python.org/3/library/itertools.html#itertools.product  
  This was used to generate all possible input combinations.

## Problem 3: Quantum Oracles

### Background

In the single-input case, a Boolean function takes one input \(x\) and returns either `True` or `False`.

There are four possible functions:

- \(f(x) = 0\) → always returns False (constant)  
- \(f(x) = 1\) → always returns True (constant)  
- \(f(x) = x\) → returns the input value (balanced)  
- \(f(x) = \neg x\) → returns the opposite of the input (balanced)  

In quantum computing, these functions are implemented as **oracles**.

The oracle must perform the transformation:

\[
U_f |x⟩|y⟩ = |x⟩ |y \oplus f(x)⟩
\]

This means the second qubit is flipped if and only if the function output is 1.

### Implementation

Each oracle is implemented using a simple quantum circuit with two qubits:

- qubit 0 → input \(|x⟩\)  
- qubit 1 → output \(|y⟩\)  

The logic is:

- For \(f(x) = 0\): do nothing (no gates needed)  
- For \(f(x) = 1\): apply an X gate to always flip the output  
- For \(f(x) = x\): use a CNOT gate so the output flips when \(x = 1\)  
- For \(f(x) = \neg x\): flip the output first, then apply CNOT  

These circuits directly implement the required transformation \(y \oplus f(x)\).

In [43]:
from qiskit import QuantumCircuit

def deutsch_oracle(name: str) -> QuantumCircuit:
    """
    Construct a 2-qubit oracle for Deutsch's algorithm.

    Qubit 0: input |x⟩
    Qubit 1: output |y⟩ (target qubit)
    """
    qc = QuantumCircuit(2, name=f"U_{name}")

    if name == "f0":
        # f(x) = 0  → do nothing
        pass

    elif name == "f1":
        # f(x) = 1  → always flip y
        qc.x(1)

    elif name == "fx":
        # f(x) = x  → flip y if x = 1
        qc.cx(0, 1)

    elif name == "fnotx":
        # f(x) = ¬x → effectively y ⊕ (1 ⊕ x)
        qc.x(1)
        qc.cx(0, 1)

    else:
        raise ValueError("Invalid oracle name: use 'f0', 'f1', 'fx', or 'fnotx'.")

    return qc


### Testing the Oracles

Each oracle is tested using different input values for \(x\) and \(y\).

This allows us to clearly see how the output qubit changes and confirms that the oracle is working correctly.

In [44]:
for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    print(f"\nOracle: {name}")
    print(oracle.draw())  # ASCII diagram



Oracle: f0
     
q_0: 
     
q_1: 
     

Oracle: f1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Oracle: fx
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

Oracle: fnotx
               
q_0: ───────■──
     ┌───┐┌─┴─┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘


### Demonstration

To verify that the oracles work correctly, each one is applied to all possible input states:

- \(x = 0\) or \(1\)  
- \(y = 0\) or \(1\)

The results show how the output qubit changes after applying the oracle.

This confirms that each oracle correctly performs the transformation:

\[
|x⟩|y⟩ \rightarrow |x⟩|y \oplus f(x)⟩
\]

### Discussion

The results confirm that each oracle behaves as expected.

- The constant functions either never flip the output qubit or always flip it.
- The balanced functions flip the output depending on the value of the input qubit.

This shows how a classical function can be encoded into a quantum circuit.

These oracles are important because they are used in Deutsch’s algorithm,
where the function is evaluated using quantum superposition.

### References

- IBM Quantum, *Deutsch's Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm  
  This was used to understand how oracles are defined and how they are used in Deutsch’s algorithm.

- Qiskit Documentation, *QuantumCircuit*:  
  https://qiskit.org/documentation/  
  This was used to understand how to construct quantum circuits and apply gates such as X and CNOT.

- Quantum Computing StackExchange, *What is a quantum oracle?*:  
  https://quantumcomputing.stackexchange.com/questions/4626  
  This helped clarify the concept of a quantum oracle and how classical functions are represented.

## Problem 4: Deutsch's Algorithm with Qiskit

Deutsch’s algorithm determines whether a single-input Boolean function
is constant or balanced using only **one oracle query**.

Classically, two evaluations may be required.
Quantum mechanically, superposition and interference allow us to determine
the global property with a single query.

Circuit steps:

1. Initialise |0⟩|1⟩
2. Apply Hadamard gates
3. Apply the oracle once
4. Apply Hadamard to the input qubit
5. Measure the input qubit

Result:
- Measurement 0 → constant
- Measurement 1 → balanced

In [46]:
from qiskit_aer import AerSimulator

sim = AerSimulator()

def deutsch_algorithm(oracle):
    qc = QuantumCircuit(2, 1)

    # Step 1: Prepare |0⟩|1⟩
    qc.x(1)

    # Step 2: Hadamards
    qc.h(0)
    qc.h(1)

    # Step 3: Oracle
    qc.append(oracle.to_gate(), [0, 1])

    # Step 4: Hadamard on input qubit
    qc.h(0)

    # Step 5: Measure input qubit
    qc.measure(0, 0)

    return qc

In [47]:
from qiskit import transpile

def classify(counts):
    return "constant" if counts.get("0", 0) > counts.get("1", 0) else "balanced"

for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    circuit = deutsch_algorithm(oracle)

    # Compile custom oracle gate into backend-supported instructions.
    compiled = transpile(circuit, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()

    print(f"{name:7s} -> {counts} -> {classify(counts)}")

f0      -> {'0': 1024} -> constant
f1      -> {'0': 1024} -> constant
fx      -> {'1': 1024} -> balanced
fnotx   -> {'1': 1024} -> balanced


The oracle imprints the function value into the phase of the superposition
(phase kickback).

After the final Hadamard gate, interference causes:

- Constructive interference at |0⟩ if the function is constant.
- Destructive interference at |0⟩ if the function is balanced.

Thus the measurement outcome deterministically reveals the function type
with only one oracle evaluation.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm

The Deutsch–Jozsa algorithm generalises Deutsch’s algorithm from one input bit
to multiple input bits.

In this assessment, the functions take **four Boolean inputs**, so there are
\(2^4 = 16\) possible input combinations.

The oracle implements:

    U_f |x⟩|y⟩ = |x⟩ |y ⊕ f(x)⟩

The Deutsch–Jozsa circuit uses:

- 4 input qubits
- 1 ancilla qubit

After applying Hadamard gates, the oracle is queried once, and then Hadamard
gates are applied again to the input register.

Interpretation of the result:

- measuring `0000` means the function is **constant**
- measuring anything else means the function is **balanced**

This works because the oracle encodes the function values into the phase of the
superposition, and interference causes the amplitudes to combine differently for
constant and balanced functions.

In [48]:
# Constant functions
def const_false(a, b, c, d):
    _ = (a, b, c, d)
    return False

def const_true(a, b, c, d):
    _ = (a, b, c, d)
    return True

# Balanced functions
def balanced_first_bit(a, b, c, d):
    _ = (b, c, d)
    return a

def balanced_parity(a, b, c, d):
    return a ^ b ^ c ^ d

In [49]:
from itertools import product
from qiskit import QuantumCircuit

def build_uf(f, n=4):
    """
    Build a Deutsch–Jozsa oracle for a Boolean function f with n inputs.

    The oracle implements:
        |x>|y> -> |x>|y XOR f(x)|

    Qubits 0..n-1 are the input register.
    Qubit n is the ancilla/target qubit.
    """
    qc = QuantumCircuit(n + 1, name="U_f")

    inputs = list(product([False, True], repeat=n))

    for x in inputs:
        if f(*x):
            # Flip qubits where input bit is 0 so all controls become on-1 controls
            for i, bit in enumerate(x):
                if bit is False:
                    qc.x(i)

            # Multi-controlled X onto ancilla
            qc.mcx(list(range(n)), n)

            # Undo the flips
            for i, bit in enumerate(x):
                if bit is False:
                    qc.x(i)

    return qc

In [50]:
def deutsch_jozsa_circuit(f, n=4):
    """
    Build the Deutsch–Jozsa circuit for an n-input Boolean function f.
    """
    oracle = build_uf(f, n)
    qc = QuantumCircuit(n + 1, n)

    # Prepare ancilla in |1>
    qc.x(n)

    # Apply Hadamard gates to all qubits
    for i in range(n + 1):
        qc.h(i)

    # Apply oracle
    qc.append(oracle.to_gate(), range(n + 1))

    # Apply Hadamard gates to input register only
    for i in range(n):
        qc.h(i)

    # Measure input register
    qc.measure(range(n), range(n))

    return qc

In [51]:
from qiskit_aer import AerSimulator

sim = AerSimulator()

def classify_deutsch_jozsa(counts):
    """
    If the measured result is all zeros, classify as constant.
    Otherwise classify as balanced.
    """
    most_common = max(counts, key=counts.get)

    if most_common == "0000":
        return "constant"
    else:
        return "balanced"

In [52]:
from qiskit import transpile

test_functions = [
    ("const_false", const_false, "constant"),
    ("const_true", const_true, "constant"),
    ("balanced_first_bit", balanced_first_bit, "balanced"),
    ("balanced_parity", balanced_parity, "balanced"),
]

for name, f, expected in test_functions:
    qc = deutsch_jozsa_circuit(f, n=4)
    # Compile custom/composite oracle instructions for Aer.
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()
    predicted = classify_deutsch_jozsa(counts)

    print(f"{name:20s} expected={expected:8s} predicted={predicted:8s} counts={counts}")

const_false          expected=constant predicted=constant counts={'0000': 1024}
const_true           expected=constant predicted=constant counts={'0000': 1024}
balanced_first_bit   expected=balanced predicted=balanced counts={'0001': 1024}
balanced_parity      expected=balanced predicted=balanced counts={'1111': 1024}
